# PPO

In [1]:
from mountaincar_utils import test_car, env_mountaincar, display_frames_as_gif, ReplayMemory
from IPython.display import HTML

In [2]:
import torch.nn as nn
import torch
# networks

class ValueNet(nn.Module):
    def __init__(self, num_states=4):
        super(ValueNet, self).__init__()
        self.fc1 = nn.Linear(num_states, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.fc2(x)
    
class ActorNet(nn.Module):
    def __init__(self, num_states=2):
        super(ActorNet, self).__init__()
        self.num_states = num_states
        self.shared = nn.Sequential(nn.Linear(self.num_states, 64), nn.ReLU())
        self.mean_layer = nn.Linear(64, 1)
        self.log_std_layer = nn.Linear(64, 1)

    def forward(self, state):
        x = self.shared(state)
        # Binds mean action to [-1.0, 1.0]
        mean = torch.tanh(self.mean_layer(x)) 
        # Keeps standard deviation positive
        std = torch.exp(torch.clamp(self.log_std_layer(x), -20.0, 2.0)) 
        return mean, std

In [3]:
import torch.nn.functional as F
import numpy as np
from torch.distributions import Normal
from torch.utils.data import DataLoader, TensorDataset

class PPOAgent:
    def __init__(self):
        self.value_net = ValueNet(num_states=2)
        self.actor_net = ActorNet(num_states=2)
        self.optim = torch.optim.AdamW(list(self.value_net.parameters())+list(self.actor_net.parameters()), lr=0.001)
        self.reset()

    def reset(self):
        self.action_value = 0.0
        self.log_prob = -1.0
        self.action_delta = 0.0

    def act(self, state, train=True):
        if isinstance(state, np.ndarray):
            if len(state.shape) == 1:
                state = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
            else:
                state = torch.tensor(state, dtype=torch.float32)
        with torch.no_grad():
            mean, std = self.actor_net(state)
            dist = Normal(mean, std)
            action_delta = dist.sample()
            self.log_prob = dist.log_prob(action_delta).unsqueeze(-1).detach().item()
            self.action_delta = action_delta.detach().item()

        self.action_value = np.clip(self.action_value + self.action_delta, -1.0, 1.0)
        env_input = np.array([self.action_value], dtype=np.float32)

        return env_input

    def learn(self, plays):
        def cum_reward(rewards, gamma=0.99):
            cum_rewards = torch.zeros_like(rewards).to(torch.float32)
            for j in range(len(rewards))[::-1]:
                cum_rewards[j] = rewards[j] + gamma * (cum_rewards[j+1] if j+1<len(rewards) else 0)
            eps = np.finfo(np.float32).eps.item()
            cum_rewards = (cum_rewards - cum_rewards.mean()) / (cum_rewards.std() + eps)
            return cum_rewards.unsqueeze(-1)

        kl_coef = 10.0
        v_coef = 10.0
        batch_size = 64
        epochs = 4

        # get data
        states, rewards, actions, old_logprobs_a, _ = plays.sample()
        rewards_cum = cum_reward(rewards)

        # Create a dataset to process in mini-batches to save memory
        dataset = TensorDataset(states, actions, old_logprobs_a, rewards_cum)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

        total_loss = 0
        for _ in range(epochs):
            for b_states, b_actions, b_old_logprobs, b_rewards_cum in loader:
                new_values = self.value_net(b_states)
                mean, std = self.actor_net(b_states)
                dist = Normal(mean, std)
                new_logprobs_a = dist.log_prob(b_actions).unsqueeze(-1)

                # Use detach on new_values for the advantage to prevent gradients flowing through value_net in policy part
                advantages = b_rewards_cum - new_values.detach()
                
                log_ratio = new_logprobs_a - b_old_logprobs
                prob_ratio = torch.exp(log_ratio)

                kl = F.kl_div(new_logprobs_a, torch.exp(b_old_logprobs), reduction='none')
                vloss = F.mse_loss(new_values, b_rewards_cum, reduction='none')

                # Standard PPO loss calculation averaged over mini-batch
                loss = (-advantages * prob_ratio + kl * kl_coef + vloss * v_coef).mean()

                self.optim.zero_grad()
                loss.backward()
                self.optim.step()
                
                total_loss += loss.detach().item()

        return total_loss / (len(loader) * epochs)


In [ ]:
# train loop

from collections import deque


epochs = 2000
state_num = 4 # 
action_num = 2
memory = ReplayMemory()
agent = PPOAgent()

scores = []
losses = []
recent_scores = deque(maxlen=100)

for e in range(epochs):
    # reset environment
    state, _ = env_mountaincar.reset()
    agent.reset()

    currState = state
    done = False

    score = 0
    tot_loss = 0
    count = 0
    tot_reward = 0.0

    # run an episode
    while not done :
        
        # choose action
        action  = agent.act(state)

        # take action on env
        state, reward, terminated, truncated, info = env_mountaincar.step(action)
        done = terminated or truncated
        
        # add to replay memory
        memory.add([currState, reward, agent.action_delta, agent.log_prob, done])

        currState = state.copy()

        # update score
        score += 1
        tot_reward += reward

        if score > 800:
            reward = -100.0  # Give a large positive reward for reaching the goal
            break
    
    # train
    loss = agent.learn(memory)

    # clear memory
    memory.clear()
    
    scores = np.append(scores, score)
    losses = np.append(losses, loss)
    recent_scores.append(tot_reward)

    if (e+1)%100 == 0:
        print(f"epoch: {e+1}, reward: {tot_reward}, scores: {score}, loss: {loss:.6f}")

    if len(recent_scores) >= 100:
        average_reward = sum(recent_scores) / 100
        if average_reward > 80.:
            print(f"Early stopping at episode {e+1} with average reward: {average_reward:.2f}")
            break


epoch: 100, reward: -56.95333927050708, scores: 801, loss: 10.512438
epoch: 200, reward: -65.38672085240599, scores: 801, loss: 7.842368
epoch: 300, reward: 80.74502148721373, scores: 304, loss: 3.944158
epoch: 400, reward: 67.8139629839344, scores: 493, loss: 4.571738
epoch: 500, reward: -55.499987064556585, scores: 801, loss: 11.293631
epoch: 600, reward: 83.64972537868043, scores: 271, loss: 4.267995
epoch: 700, reward: 52.375172307482465, scores: 559, loss: 8.098299
epoch: 800, reward: 74.845657687109, scores: 452, loss: 4.094871
epoch: 900, reward: 88.46215760767801, scores: 193, loss: 7.467312
epoch: 1000, reward: 53.694809609076636, scores: 753, loss: 4.080442


In [10]:
frames = test_car(env_mountaincar, 900, agent=agent)
anim = display_frames_as_gif(frames)
HTML(anim.to_jshtml())

371
